# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
import chromadb
from lib.tooling import tool
from lib.vector_db import VectorStore

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.
    """
    vector_store = VectorStore(chroma_collection=collection)
    results = vector_store.query(query_texts=[query], n_results=5)
    return results.get("documents", [[]])[0]

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from lib.evaluation import AgentEvaluator
from lib.tooling import tool
from tavily import TavilyClient

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
        args:
        - question: original question from user
        - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    agent_evaluator = AgentEvaluator(api_key=OPENAI_API_KEY)
    evaluation = agent_evaluator.evaluate_retrieval(question, retrieved_docs)
    return {
        "useful": evaluation.task_completion.task_completed,
        "description": evaluation.feedback,
        "score": evaluation.overall_score,
    }


def tavily_web_search(
    question: str,
    max_results: int = 5,
) -> dict:
    """
    Search the web for video-game industry information using Tavily.

    Args:
        question: A question about the video game industry.
        max_results: Maximum number of search results to return.
    """
    tavily_client = TavilyClient(
        api_key=os.getenv("TAVILY_API_KEY")
    )

    return tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=max_results,
        include_answer=False,
        include_raw_content=False,
    )

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.
@tool
def game_web_search(
    question: str,
    max_results: int = 5,
    confidence_threshold: float = 0.5,
) -> list[dict]:
    retrieved_docs = retrieve_game(question)

    evaluation = evaluate_retrieval(
        question=question,
        retrieved_docs=retrieved_docs,
    )

    internal_results = [
        {
            "content": doc,
            "source_type": "internal",
        }
        for doc in retrieved_docs
    ]

    useful = evaluation.get("useful", False)
    score = evaluation.get("score", 0.0)

    if useful and score >= confidence_threshold:
        return internal_results

    web_response = tavily_web_search(
        question=question,
        max_results=max_results,
    )

    raw_web_results = web_response.get("results", [])

    web_results = [
        {
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content", ""),
            "score": result.get("score"),
            "published_date": result.get("published_date"),
            "source_type": "web",
        }
        for result in raw_web_results
    ]

    if useful and internal_results:
        return internal_results + web_results

    return web_results

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, game_web_search, evaluate_retrieval],
    instructions=(
        "You are an Agentic RAG assistant that can intelligently decide which tools to use "
        "to answer user questions. Reason about the response, change the query and call the tool again if needed "
        "in order to get better results. Always explain your reasoning for tool selection and provide comprehensive answers."
    ),
    temperature=0.0,
)

In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
from lib.utils import print_result

query = "When Pokémon Gold and Silver was released?"
run_1 = agentic_rag.invoke(
    query=query,
    session_id="pokemon",
)
print_result(query, run_1)


Question: When Pokémon Gold and Silver was released?
Tool: game_web_search

Answer:
Pokémon Gold and Silver were released on the following dates:

- **Japan**: November 21, 1999
- **North America**: October 15, 2000
- **Australia**: October 13, 2000
- **Europe**: April 6, 2001
- **South Korea**: April 24, 2002

These games were the first installments in the second generation of the Pokémon series and were developed for the Game Boy Color. They introduced 100 new species of Pokémon and expanded the gameplay significantly compared to their predecessors. 

For more detailed information, you can check the [Wikipedia page](https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver) or [Bulbapedia](https://bulbapedia.bulbagarden.net/wiki/Pok%C3%A9mon_Gold_and_Silver_Versions).



In [9]:
query = "Which one was the first 3D platformer Mario game?"
run_2 = agentic_rag.invoke(
    query=query,
    session_id="mario",
)
print_result(query, run_2)


Question: Which one was the first 3D platformer Mario game?
Tool: retrieve_game
Tool: evaluate_retrieval

Answer:
The first 3D platformer Mario game is **Super Mario 64**, released in 1996 for the Nintendo 64. This game was groundbreaking and set new standards for the platforming genre, featuring Mario's quest to rescue Princess Peach in a fully realized 3D environment. 

While the search retrieved some other games, only Super Mario 64 was relevant to your query about 3D platformers.



In [10]:
query = "Was Mortal Kombat X realeased for Playstation 5?"
run_3 = agentic_rag.invoke(
    query=query,
    session_id="mortal_kombat",
)
print_result(query, run_3)


Question: Was Mortal Kombat X realeased for Playstation 5?
Tool: game_web_search
Tool: evaluate_retrieval
Tool: retrieve_game
Tool: game_web_search

Answer:
Mortal Kombat X was not released specifically for the PlayStation 5, but it is playable on the PS5. The game was originally released for the PlayStation 4 on April 14, 2015. However, due to backward compatibility, players can run the PS4 version of Mortal Kombat X on the PS5. 

It's important to note that while the game is playable on the PS5, some features that were available on the PS4 may not be present when playing on the newer console. For optimal performance, it is recommended to keep the PS5 updated with the latest system software.

Additionally, Mortal Kombat X is included in the PlayStation Plus Collection, which allows subscribers to access it on the PS5. This means that if you have a PlayStation Plus subscription, you can download and play Mortal Kombat X on your PS5 without needing to own the PS4 version separately.

F

### (Optional) Advanced

In [11]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes
query = "What is Rockstar Games currently working on?"
run_4 = agentic_rag.invoke(
    query=query,
    session_id="rockstar",
)
print_result(query, run_4)
# Please check out main.py


Question: What is Rockstar Games currently working on?
Tool: retrieve_game
Tool: game_web_search
Tool: evaluate_retrieval

Answer:
Rockstar Games is currently working on several projects, with the most notable being the remake of the **Max Payne 1 and 2** games. This project is a collaboration with **Remedy Entertainment**, the original creators of the Max Payne series. Here are the key details:

1. **Max Payne Remake**: 
   - The remake is being developed using Remedy's Northlight Engine, which powered their recent titles like *Control* and *Alan Wake 2*. This indicates a significant upgrade in graphics and gameplay mechanics.
   - The project is not just a remaster; it aims to create a modern recreation of both games, merging them into a single continuous experience. This means that the narrative flow will be enhanced, potentially adding new content and scenes to connect the two stories more seamlessly.
   - Rockstar is heavily involved in the development process, moving beyond just